# BIFlow — Notebook 2 : Analyste BI

**Objectif métier :** où observe-t-on des évolutions inhabituelles du volume de demandes ou des délais de clôture, et quelles zones ou catégories méritent une investigation ?

Le notebook calcule les KPI, tendances, anomalies et preuves avec des règles déterministes. Un seul modèle local, Llama 3.2 3B via Ollama, rédige ensuite un bref résumé qualitatif à partir des insights agrégés. Les chiffres et décisions du rapport restent ceux des calculs vérifiables ; aucune API distante ni clé n’est utilisée.

## Architecture et fichiers nécessaires

Ordre de lecture : chargement/contrôles → descriptif → tendances → anomalies → fiches d’insights → synthèse et exports. Le fichier principal est `data/processed/nyc311_clean.csv`, produit par `notebooks/01_etl_kpi.ipynb`. Le catalogue `data/processed/kpi_catalog.csv` est chargé s’il existe, pour conserver le contexte sémantique. Le notebook ne télécharge aucune donnée.

Chaque section explique l’objectif, son utilité pour l’analyste, l’interprétation attendue et les limites. Les comparaisons mensuelles nécessitent plusieurs mois ; les groupes de délais ont un effectif minimal explicite. Les dates de fermeture, et non le statut textuel, définissent ici une demande clôturée.

## 1. Chargement et contrôles

On charge la sortie nettoyée sans la modifier et on travaille sur une copie analytique. Les contrôles de colonnes optionnelles permettent de désactiver proprement une analyse si une dimension manque. Les dates sont converties dans la copie et les périodes effectivement couvertes sont affichées.

**À interpréter :** une période concentrée sur un mois, beaucoup de demandes sans fermeture, ou des délais absents réduisent fortement la portée des comparaisons. L’absence d’une colonne facultative n’est pas compensée par une colonne inventée.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)
plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "data/processed/nyc311_clean.csv").exists()), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
CLEAN_PATH = PROJECT_ROOT / "data/processed/nyc311_clean.csv"
CATALOG_PATH = PROJECT_ROOT / "data/processed/kpi_catalog.csv"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
SAMPLE_METADATA_PATH = PROJECT_ROOT / "data/raw/nyc311_sample_metadata.json"
if not CLEAN_PATH.exists():
    raise FileNotFoundError(
        f"Fichier requis absent : {CLEAN_PATH}. Exécutez d’abord toutes les cellules "
        "de notebooks/01_etl_kpi.ipynb avec le CSV placé dans data/raw/nyc311_sample.csv."
    )
df_source = pd.read_csv(CLEAN_PATH, low_memory=False)
df = df_source.copy(deep=True)
catalog = pd.read_csv(CATALOG_PATH) if CATALOG_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
sample_metadata = json.loads(SAMPLE_METADATA_PATH.read_text(encoding="utf-8")) if SAMPLE_METADATA_PATH.exists() else None
print("Fichier nettoyé :", CLEAN_PATH, "| dimensions :", df.shape)
print("Catalogue KPI :", "disponible" if catalog is not None else "absent (facultatif)")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
print("Métadonnées d’échantillonnage :", "disponibles" if sample_metadata is not None else "absentes")
display(df.head())
display(df.dtypes.rename("type initial").to_frame())
print("Colonnes réellement disponibles :", df.columns.tolist())

required = {"volume_mensuel": ["created_date"],
            "delais": ["created_date", "closed_date", "resolution_days"],
            "par_type": ["complaint_type"], "par_arrondissement": ["borough"],
            "par_agence": ["agency"]}
availability = pd.DataFrame([{"analyse": name, "disponible": all(c in df.columns for c in cols),
                              "colonnes_manquantes": [c for c in cols if c not in df.columns]}
                             for name, cols in required.items()])
display(availability)
for col in ["created_date", "closed_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
if "resolution_days" in df.columns:
    df["resolution_days"] = pd.to_numeric(df["resolution_days"], errors="coerce")
    df.loc[~np.isfinite(df["resolution_days"]), "resolution_days"] = np.nan
    df.loc[df["resolution_days"] < 0, "resolution_days"] = np.nan

if "created_date" in df.columns:
    print("Dates de création valides :", int(df["created_date"].notna().sum()), "/", len(df))
    print("Période couverte :", df["created_date"].min(), "→", df["created_date"].max())
    df["month"] = df["created_date"].dt.to_period("M").astype("string")
else:
    df["month"] = pd.Series(pd.NA, index=df.index, dtype="string")
if "closed_date" in df.columns:
    print("Dates de fermeture valides :", int(df["closed_date"].notna().sum()))
if "resolution_days" in df.columns:
    print("Délais numériques valides :", int(df["resolution_days"].notna().sum()))
print("Limite : les demandes sans date de fermeture peuvent être en cours ; elles ne sont pas traitées comme clôturées.")

Fichier nettoyé : D:\Analyst_Agent\.tmp_jupyter\audit_20261024\data\processed\nyc311_clean.csv | dimensions : (10000, 49)
Catalogue KPI : disponible
Métadonnées d’échantillonnage : disponibles
Métadonnées d’échantillonnage : disponibles
Métadonnées d’échantillonnage : disponibles
Métadonnées d’échantillonnage : disponibles
Métadonnées d’échantillonnage : disponibles
Métadonnées d’échantillonnage : disponibles


,unique_key,created_date,closed_date,agency,agency_name,complaint_type,descriptor,descriptor_2,location_type,incident_zip,incident_address,street_name,cross_street_1,cross_street_2,intersection_street_1,intersection_street_2,address_type,city,landmark,facility_type,status,due_date,resolution_description,resolution_action_updated_date,community_board,council_district,police_precinct,bbl,borough,x_coordinate_state_plane,y_coordinate_state_plane,open_data_channel_type,park_facility_name,park_borough,vehicle_type,taxi_company_borough,taxi_pick_up_location,bridge_highway_name,bridge_highway_direction,road_ramp,bridge_highway_segment,latitude,longitude,location,temporal_inconsistency,resolution_days,duration_suspicious,created_month,created_year
0,69243362,2026-06-04 15:32:09+00:00,2026-07-13 13:34:04+00:00,DPR,Department of Parks and Recreation,Dead/Dying Tree,Planted More Than 2 Years Ago,NaN,Street,11209.0,602 76 STREET,76 STREET,6 AVENUE,7 AVENUE,6 AVENUE,7 AVENUE,ADDRESS,BROOKLYN,76 STREET,NaN,Closed,NaN,NYC Parks visited the site and inspected the condition. No work is necessary at this time.,2026-07-13T13:34:09.000,10 BROOKLYN,47.0,Precinct 68,3.059530e+09,BROOKLYN,978609.0,167747.0,PHONE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.627101,-74.020322,POINT (-74.02032177502 40.62710148779),False,38.917998,False,2026-06,2026
1,69236046,2026-06-04 15:32:22+00:00,2026-08-03 15:35:09+00:00,DOHMH,Department of Health and Mental Hygiene,Mobile Food Vendor,Food Temperature,Bad Food-Improper Temperature,Mobile Food Vendor,10036.0,199 WEST 41 STREET,WEST 41 STREET,BROADWAY,7 AVENUE,BROADWAY,7 AVENUE,ADDRESS,NEW YORK,WEST 41 STREET,NaN,Closed,NaN,"The NYC Health Department has responded to your service request. For inspection results, please call (212) 676-1600 ...",2026-08-03T15:35:12.000,05 MANHATTAN,3.0,Precinct 14,NaN,MANHATTAN,987764.0,214464.0,MOBILE,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.755330,-73.987316,POINT (-73.987316400971 40.755329950391),False,60.001933,False,2026-06,2026
2,69240541,2026-06-04 15:32:44+00:00,2026-06-10 12:57:10+00:00,DOT,Department of Transportation,Sidewalk Condition,Sidewalk Violation,Sidewalk Re-inspection Request,Sidewalk,10459.0,1282 HOE AVENUE,HOE AVENUE,FREEMAN STREET,JENNINGS STREET,FREEMAN STREET,JENNINGS STREET,ADDRESS,BRONX,HOE AVENUE,NaN,Closed,NaN,The Department of Transportation has approved the sidewalk re-inspection request. The property owner will be notifie...,2026-06-10T12:57:14.000,03 BRONX,17.0,Precinct 42,2.029870e+09,BRONX,1014508.0,241909.0,PHONE,Unspecified,BRONX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.830608,-73.890662,POINT (-73.890661741454 40.830607926513),False,5.891968,False,2026-06,2026
3,69234291,2026-06-04 15:32:58+00:00,2026-06-11 09:05:00+00:00,DOT,Department of Transportation,Street Condition,Pothole,NaN,NaN,10472.0,NaN,NaN,NaN,NaN,WESTCHESTER AVENUE,WHITE PLAINS ROAD,INTERSECTION,BRONX,NaN,NaN,Closed,NaN,The Department of Transportation inspected this complaint and repaired the problem.,2026-06-11T09:05:00.000,09 BRONX,18.0,Precinct 43,NaN,BRONX,1022238.0,242763.0,UNKNOWN,Unspecified,BRONX,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.832922,-73.862724,POINT (-73.862724342239 40.832922043106),False,6.730579,False,2026-06,2026
4,69233477,2026-06-04 15:33:00+00:00,2026-06-04 18:35:00+00:00,DEP,Department of Environmental Protection,Sewer,Sewer Odor (SA2),SA2,NaN,11436.0,147-20 119 AVENUE,119 AVENUE,147 ST,BEND,NaN,NaN,ADDRESS,JAMAICA,NaN,NaN,Closed,NaN,The Department of Environment Protection inspected your complaint but could not find the problem you reported. If th...,2026-06-04T18:35:00.000,12 QUEENS,28.0,Precinct 113,4.120280e+09,QUEENS,1041811.0,186783.0,PHONE,Unspecified,QUEENS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.679166,-73.792474,POINT (-73.792473675144 40.679165800832),False,0.126389,False,2026-06,2026


,type initial
unique_key,int64
created_date,str
closed_date,str
agency,str
agency_name,str
complaint_type,str
descriptor,str
descriptor_2,str
location_type,str
incident_zip,float64


Colonnes réellement disponibles : ['unique_key', 'created_date', 'closed_date', 'agency', 'agency_name', 'complaint_type', 'descriptor', 'descriptor_2', 'location_type', 'incident_zip', 'incident_address', 'street_name', 'cross_street_1', 'cross_street_2', 'intersection_street_1', 'intersection_street_2', 'address_type', 'city', 'landmark', 'facility_type', 'status', 'due_date', 'resolution_description', 'resolution_action_updated_date', 'community_board', 'council_district', 'police_precinct', 'bbl', 'borough', 'x_coordinate_state_plane', 'y_coordinate_state_plane', 'open_data_channel_type', 'park_facility_name', 'park_borough', 'vehicle_type', 'taxi_company_borough', 'taxi_pick_up_location', 'bridge_highway_name', 'bridge_highway_direction', 'road_ramp', 'bridge_highway_segment', 'latitude', 'longitude', 'location', 'temporal_inconsistency', 'resolution_days', 'duration_suspicious', 'created_month', 'created_year']


,analyse,disponible,colonnes_manquantes
0,volume_mensuel,True,[]
1,delais,True,[]
2,par_type,True,[]
3,par_arrondissement,True,[]
4,par_agence,True,[]


Dates de création valides : 10000 / 10000
Période couverte : 2026-06-04 15:32:09+00:00 → 2026-10-14 09:05:00+00:00
Dates de fermeture valides : 9291
Délais numériques valides : 9285
Limite : les demandes sans date de fermeture peuvent être en cours ; elles ne sont pas traitées comme clôturées.


C:\Users\HP\AppData\Local\Temp\ipykernel_8264\1813660046.py:68: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df["month"] = df["created_date"].dt.to_period("M").astype("string")


## 2. Analyse descriptive

Les ventilations répondent à la question « quoi/où/qui ? ». Elles comptent les lignes de l’échantillon et excluent une dimension manquante uniquement du tableau correspondant. Pour le volume mensuel, les métadonnées de l’extrait fournissent aussi les totaux agrégés officiels du portail ; le notebook les utilise lorsqu’ils concordent avec les mois présents. Les ventilations par catégorie et les délais restent calculés sur l’échantillon de 10 000 lignes. Pour les délais, on limite la population aux demandes ayant une date de fermeture et un délai positif ou nul exploitable ; médiane et quantiles complètent la moyenne.

**À interpréter :** les tableaux et graphiques décrivent l’échantillon. Un volume élevé ne mesure ni le taux de problème par habitant ni la performance de l’agence. Sans dénominateur de population, on ne compare pas les arrondissements comme des taux.

In [2]:
def volume_table(frame, column):
    if column not in frame.columns:
        print(f"Analyse ignorée : colonne facultative {column!r} absente.")
        return None
    return (frame.loc[frame[column].notna()].groupby(column).size()
            .rename("request_count").sort_values(ascending=False).reset_index())

volume_by_type = volume_table(df, "complaint_type")
volume_by_borough = volume_table(df, "borough")
volume_by_agency = volume_table(df, "agency")
for table in [volume_by_type, volume_by_borough, volume_by_agency]:
    if table is not None:
        display(table.head(15))

monthly_volume = None
if df["month"].notna().any():
    monthly_volume = (df.dropna(subset=["month"]).groupby("month").size()
                      .rename("request_count").reset_index())
    monthly_volume["sample_count"] = monthly_volume["request_count"]
    monthly_volume["count_basis"] = "sample rows"
    if sample_metadata is not None:
        population_counts = sample_metadata.get("population_monthly_counts", {})
        if set(monthly_volume["month"]).issubset(population_counts):
            monthly_volume["request_count"] = monthly_volume["month"].map(population_counts).astype(int)
            monthly_volume["count_basis"] = "official monthly aggregate; sample used for dimensions/delays"
    monthly_volume["month_start"] = pd.PeriodIndex(monthly_volume["month"], freq="M").to_timestamp()
    monthly_volume = monthly_volume.sort_values("month_start").reset_index(drop=True)
    display(monthly_volume)
else:
    print("Volume mensuel non calculable : aucune date de création valide.")

closed_mask = pd.Series(False, index=df.index)
if "closed_date" in df.columns:
    closed_mask = df["closed_date"].notna()
valid_delay_mask = closed_mask & df["resolution_days"].notna() if "resolution_days" in df.columns else closed_mask
valid_delays = df.loc[valid_delay_mask, "resolution_days"].dropna() if "resolution_days" in df.columns else pd.Series(dtype=float)
delay_distribution = pd.DataFrame([{
    "closed_with_valid_delay_n": int(valid_delay_mask.sum()),
    "mean_days": valid_delays.mean() if len(valid_delays) else np.nan,
    "median_days": valid_delays.median() if len(valid_delays) else np.nan,
    "p25_days": valid_delays.quantile(.25) if len(valid_delays) else np.nan,
    "p75_days": valid_delays.quantile(.75) if len(valid_delays) else np.nan,
    "p90_days": valid_delays.quantile(.90) if len(valid_delays) else np.nan,
}])
display(delay_distribution)
print("Moyenne = sensible aux longues durées ; médiane/quantiles décrivent mieux une distribution asymétrique.")

if monthly_volume is not None:
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(monthly_volume["month_start"], monthly_volume["request_count"], marker="o")
    ax.set(title="Volume mensuel officiel" + (" — octobre partiel" if sample_metadata is not None and "2026-10" in sample_metadata.get("partial_months", []) else ""), xlabel="Mois de création", ylabel="Nombre de demandes")
    plt.xticks(rotation=45); plt.tight_layout(); plt.show()
if len(valid_delays):
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(valid_delays, bins=35, color="steelblue")
    ax.set(title="Délais de clôture valides", xlabel="Jours calendaires", ylabel="Nombre de demandes")
    plt.tight_layout(); plt.show()
print("Limite : les demandes sans fermeture/délai valide n’entrent pas dans la distribution des délais.")

,complaint_type,request_count
0,Illegal Parking,1552
1,Noise - Residential,1028
2,Noise - Street/Sidewalk,931
3,Blocked Driveway,496
4,UNSANITARY CONDITION,349
5,Damaged Tree,273
6,Street Condition,271
7,Abandoned Vehicle,203
8,Dirty Condition,203
9,Noise - Vehicle,202


,borough,request_count
0,BROOKLYN,3337
1,QUEENS,2470
2,BRONX,1907
3,MANHATTAN,1851
4,STATEN ISLAND,426
5,Unspecified,9


,agency,request_count
0,NYPD,5023
1,HPD,1433
2,DSNY,777
3,DOT,720
4,DPR,633
5,DEP,608
6,DOB,317
7,DOHMH,215
8,DHS,113
9,TLC,77


,month,request_count,sample_count,count_basis,month_start
0,2026-06,334834,2242,official monthly aggregate; sample used for dimensions/delays,2026-06-01
1,2026-07,343133,2298,official monthly aggregate; sample used for dimensions/delays,2026-07-01
2,2026-08,328918,2203,official monthly aggregate; sample used for dimensions/delays,2026-08-01
3,2026-09,323627,2167,official monthly aggregate; sample used for dimensions/delays,2026-09-01
4,2026-10,162811,1090,official monthly aggregate; sample used for dimensions/delays,2026-10-01


,closed_with_valid_delay_n,mean_days,median_days,p25_days,p75_days,p90_days
0,9285,4.106831,0.181956,0.04412,2.76963,11.118118


Moyenne = sensible aux longues durées ; médiane/quantiles décrivent mieux une distribution asymétrique.


C:\Users\HP\AppData\Local\Temp\ipykernel_8264\3556863625.py:52: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.xticks(rotation=45); plt.tight_layout(); plt.show()


Limite : les demandes sans fermeture/délai valide n’entrent pas dans la distribution des délais.


C:\Users\HP\AppData\Local\Temp\ipykernel_8264\3556863625.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Renforcement statistique — fonctions d'analyse réutilisables

Cette extension conserve les analyses, fonctions, chemins et sorties du notebook existant. Elle ajoute des contrôles préalables, comparaisons et détections explicables, sans recalculer les KPI définis par le Notebook 1. Le catalogue existant fournit notamment les définitions de `monthly_request_volume`, `median_resolution_by_group` et `median_resolution_days`. Le rapport qualité n'est exploité que s'il correspond aux lignes actuellement chargées ; un rapport d'un autre jeu est signalé comme non concordant.

**Méthodes et limites :** le Z-score classique utilise moyenne/écart-type d'un historique antérieur ; le score MAD robuste utilise médiane/MAD. Tous deux s'abstiennent si l'historique est trop court ou la dispersion nulle. Les seuils statistiques servent au repérage et ne sont pas des objectifs métier.

In [3]:
# Contrôles additionnels sur le schéma réel et les sources de qualité disponibles.
import importlib.util
import hashlib
from datetime import datetime, timezone

def check_columns(frame, columns):
    """Retourne les colonnes requises disponibles et absentes, sans modifier frame."""
    present = [c for c in columns if c in frame.columns]
    missing = [c for c in columns if c not in frame.columns]
    return {"present": present, "missing": missing, "ready": not missing}

def data_quality_precheck(frame):
    """Résume types, manquants, dates, délais et couverture temporelle exploitable."""
    result = {"rows": int(len(frame)), "missing_percent": (frame.isna().mean()*100).round(2).to_dict()}
    for col in ["created_date", "closed_date"]:
        if col in frame:
            parsed = pd.to_datetime(frame[col], errors="coerce", utc=True)
            result[col] = {"valid_n": int(parsed.notna().sum()), "invalid_or_missing_n": int(parsed.isna().sum()),
                           "min": None if parsed.notna().sum()==0 else str(parsed.min()),
                           "max": None if parsed.notna().sum()==0 else str(parsed.max())}
    if "resolution_days" in frame:
        numeric = pd.to_numeric(frame["resolution_days"], errors="coerce")
        result["resolution_days"] = {"valid_finite_nonnegative_n": int((np.isfinite(numeric)&numeric.ge(0)).sum()),
                                     "invalid_or_negative_n": int((~np.isfinite(numeric)|numeric.lt(0)).sum())}
    return result

precheck = data_quality_precheck(df)
print("Contrôle pré-analytique :")
display(pd.DataFrame([{"contrôle": k, "résultat": str(v)} for k,v in precheck.items()]))

quality_candidates = [PROJECT_ROOT/"data/processed/data_quality_report.json",
                      PROJECT_ROOT/"biflow_analyste/data_quality_report.json"]
quality_report_path = next((p for p in quality_candidates if p.exists()), None)
quality_report = None
quality_report_status = "absent"
if quality_report_path is not None:
    loaded_quality = json.loads(quality_report_path.read_text(encoding="utf-8"))
    reported_rows = loaded_quality.get("lignes_propres", loaded_quality.get("rows", loaded_quality.get("row_count")))
    if reported_rows is None or int(reported_rows) == len(df):
        quality_report, quality_report_status = loaded_quality, "disponible_et_concordant"
    else:
        quality_report_status = f"présent_mais_non_concordant ({reported_rows} lignes rapportées vs {len(df)} lignes chargées)"
print("Rapport qualité :", quality_report_status, "|", quality_report_path)

# Lecture de quelques champs bruts pour vérifier les doublons potentiellement à l'origine de pics.
# Lecture seule ; la règle de dédoublonnage du Notebook 1 n'est pas reproduite ici.
RAW_PATH = PROJECT_ROOT/"data/raw/nyc311_sample.csv"
raw_duplicate_by_month = pd.DataFrame(columns=["month", "raw_duplicate_rows"])
raw_quality_status = "source_brute_absente"
if RAW_PATH.exists() and "unique_key" in df.columns and "created_date" in df.columns:
    raw_check = pd.read_csv(RAW_PATH, usecols=lambda c: c in {"unique_key", "created_date"}, low_memory=False)
    raw_check["created_date"] = pd.to_datetime(raw_check["created_date"], errors="coerce", utc=True)
    raw_check["month"] = raw_check["created_date"].dt.to_period("M").astype("string")
    duplicate_mask = raw_check["unique_key"].notna() & raw_check.duplicated("unique_key", keep="first")
    raw_duplicate_by_month = (raw_check.loc[duplicate_mask].groupby("month").size()
                              .rename("raw_duplicate_rows").reset_index())
    raw_quality_status = "contrôle_clé_raw_effectué"
print("Contrôle doublons raw :", raw_quality_status, "| lignes répétées :", int(raw_duplicate_by_month["raw_duplicate_rows"].sum()) if len(raw_duplicate_by_month) else 0)

Contrôle pré-analytique :


,contrôle,résultat
0,rows,10000
1,missing_percent,"{'unique_key': 0.0, 'created_date': 0.0, 'closed_date': 7.09, 'agency': 0.0, 'agency_name': 0.0, 'complaint_type': 0..."
2,created_date,"{'valid_n': 10000, 'invalid_or_missing_n': 0, 'min': '2026-06-04 15:32:09+00:00', 'max': '2026-10-14 09:05:00+00:00'}"
3,closed_date,"{'valid_n': 9291, 'invalid_or_missing_n': 709, 'min': '2026-06-04 15:40:00+00:00', 'max': '2026-10-15 20:47:40+00:00'}"
4,resolution_days,"{'valid_finite_nonnegative_n': 9285, 'invalid_or_negative_n': 715}"


Rapport qualité : présent_mais_non_concordant (41165 lignes rapportées vs 10000 lignes chargées) | D:\Analyst_Agent\.tmp_jupyter\audit_20261024\biflow_analyste\data_quality_report.json
Contrôle doublons raw : contrôle_clé_raw_effectué | lignes répétées : 0


C:\Users\HP\AppData\Local\Temp\ipykernel_8264\2433644329.py:53: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  raw_check["month"] = raw_check["created_date"].dt.to_period("M").astype("string")


## 3. Tendances temporelles

Une comparaison de tendance demande une série mensuelle triée et plusieurs périodes complètes. Nous calculons une pente linéaire descriptive du volume et une variation première/dernière période, seulement à partir de trois mois observés. Cette méthode simple évite une décomposition saisonnière inadaptée à une série courte ou irrégulière. Les délais mensuels sont calculés sur les seules demandes fermées avec délai valide.

**À interpréter :** la pente indique une direction moyenne, pas une cause ni une prévision. Un mois partiel à la fin de l’extrait peut artificiellement faire baisser le volume ; la couverture doit être examinée avant toute comparaison.

In [4]:
def temporal_trend(series, value_col, metric_name, min_periods=3):
    if series is None or len(series) < min_periods:
        return {"metric": metric_name, "n_periods": 0 if series is None else len(series),
                "direction": "insufficient_data", "slope_per_month": np.nan,
                "first_value": np.nan, "last_value": np.nan, "change_percent": np.nan}
    y = series[value_col].astype(float).to_numpy()
    x = np.arange(len(y), dtype=float)
    slope = float(np.polyfit(x, y, 1)[0])
    first, last = float(y[0]), float(y[-1])
    change = (last - first) / abs(first) * 100 if first != 0 else np.nan
    direction = "up" if slope > 0 else "down" if slope < 0 else "flat"
    return {"metric": metric_name, "n_periods": len(series), "direction": direction,
            "slope_per_month": slope, "first_value": first, "last_value": last,
            "change_percent": change}

volume_trend = temporal_trend(monthly_volume, "request_count", "monthly_request_volume")
monthly_delay = None
if "month" in df.columns and "resolution_days" in df.columns:
    delay_population = df.loc[valid_delay_mask & df["month"].notna()]
    if len(delay_population):
        monthly_delay = (delay_population.groupby("month")["resolution_days"]
                         .agg(median_days="median", sample_size="size").reset_index())
        monthly_delay["month_start"] = pd.PeriodIndex(monthly_delay["month"], freq="M").to_timestamp()
        monthly_delay = monthly_delay.sort_values("month_start").reset_index(drop=True)
delay_trend = temporal_trend(monthly_delay, "median_days", "monthly_median_resolution_days")
trend_results = pd.DataFrame([volume_trend, delay_trend])
display(trend_results)
if monthly_delay is not None:
    display(monthly_delay)
print("Observations temporelles : volume =", volume_trend["n_periods"], "mois ; délai =", delay_trend["n_periods"], "mois.")
print("Limite : sans au moins trois mois, aucune direction de tendance n’est affirmée. Contrôler les mois incomplets.")

,metric,n_periods,direction,slope_per_month,first_value,last_value,change_percent
0,monthly_request_volume,5,down,-36355.200000,334834.000000,162811.000000,-51.375607
1,monthly_median_resolution_days,5,down,-0.039928,0.336331,0.122373,-63.615403


,month,median_days,sample_size,month_start
0,2026-06,0.336331,2147,2026-06-01
1,2026-07,0.112685,2212,2026-07-01
2,2026-08,0.347986,2095,2026-08-01
3,2026-09,0.141319,1951,2026-09-01
4,2026-10,0.122373,880,2026-10-01


Observations temporelles : volume = 5 mois ; délai = 5 mois.
Limite : sans au moins trois mois, aucune direction de tendance n’est affirmée. Contrôler les mois incomplets.


## Comparaisons de groupes, tendances et anomalies par deux méthodes

Les tableaux comparent des médianes de délai avec leurs effectifs et rangs ; les échantillons sous le minimum sont marqués et non promus comme conclusion. La tendance fournit variations absolues/relatives et moyenne mobile trois périodes. Les anomalies utilisent une période de référence antérieure, afin que la valeur testée ne participe pas à sa propre référence. Z-score et MAD sont calculés sur les mêmes observations. Une MAD ou un écart-type nul produit un statut non calculable, jamais un faux score.

La saisonnalité STL mensuelle exige au minimum deux cycles annuels complets (24 mois), un index mensuel régulier et aucune valeur manquante. Les données ne sont pas interpolées. `statsmodels` est optionnel : son absence ne bloque pas les analyses de base.

In [5]:
# Définitions des fonctions statistiques et des analyses comparatives.
MIN_REFERENCE_N = 6
Z_THRESHOLD = 3.0
MAD_THRESHOLD = 3.5
MIN_GROUP_N = 30
MIN_STL_MONTHS = 24
RULE_VERSION = "biflow-bi-analyst-rules-1.0"

def detect_zscore(reference, observed, threshold=Z_THRESHOLD, min_n=MIN_REFERENCE_N, period=None):
    """Teste observed contre un historique antérieur ; renvoie statut et preuves Z-score."""
    x = pd.to_numeric(pd.Series(reference), errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    base = {"observed": float(observed) if pd.notna(observed) else np.nan,
            "reference_mean": np.nan, "reference_std": np.nan, "score": np.nan,
            "threshold": float(threshold), "direction": "none", "period": period,
            "reference_n": int(len(x)), "reference_period": None, "status": "not_calculable"}
    if len(x) < min_n or not np.isfinite(observed):
        base["status"] = "insufficient_history_or_invalid_value"; return base
    mean, std = float(x.mean()), float(x.std(ddof=1))
    base.update(reference_mean=mean, reference_std=std, reference_period=f"{len(x)} valeurs précédentes")
    if not np.isfinite(std) or std == 0:
        base["status"] = "zero_or_invalid_std"; return base
    score = (float(observed)-mean)/std
    base.update(score=score, direction="positive" if score>=threshold else "negative" if score<=-threshold else "none",
                status="anomaly" if abs(score)>=threshold else "normal")
    return base

def detect_mad(reference, observed, threshold=MAD_THRESHOLD, min_n=MIN_REFERENCE_N, period=None):
    """Teste observed contre un historique antérieur avec score robuste médiane/MAD."""
    x = pd.to_numeric(pd.Series(reference), errors="coerce").to_numpy(dtype=float)
    x = x[np.isfinite(x)]
    median = float(np.median(x)) if len(x) else np.nan
    mad = float(np.median(np.abs(x-median))) if len(x) else np.nan
    result = {"observed": float(observed) if pd.notna(observed) else np.nan,
              "reference_median": median, "mad": mad, "score": np.nan,
              "threshold": float(threshold), "direction": "none", "period": period,
              "reference_n": int(len(x)), "reference_period": f"{len(x)} valeurs précédentes" if len(x) else None,
              "status": "not_calculable"}
    if len(x)<min_n or not np.isfinite(observed):
        result["status"]="insufficient_history_or_invalid_value"; return result
    if not np.isfinite(mad) or mad==0:
        result["status"]="zero_or_invalid_mad"; return result
    score=.6745*(float(observed)-median)/mad
    result.update(score=score, direction="positive" if score>=threshold else "negative" if score<=-threshold else "none",
                  status="anomaly" if abs(score)>=threshold else "normal")
    return result

def compare_group_delays(frame, dimensions, min_n=MIN_GROUP_N):
    """Compare KPI de délai existant par dimension, en affichant effectif et rang."""
    if "resolution_days" not in frame or "closed_date" not in frame:
        return pd.DataFrame()
    pop=frame.loc[frame["closed_date"].notna() & frame["resolution_days"].notna() & (frame["resolution_days"]>=0)].copy()
    tables=[]
    formula=(catalog.loc[catalog["kpi_name"]=="median_resolution_by_group","formula"].iloc[0]
             if catalog is not None and "kpi_name" in catalog and (catalog["kpi_name"]=="median_resolution_by_group").any()
             else "Définition absente du catalogue ; comparaison descriptive du champ resolution_days")
    for dim in dimensions:
        if dim not in pop: continue
        part=(pop.dropna(subset=[dim]).groupby(dim)["resolution_days"]
              .agg(sample_size="size",mean_days="mean",median_days="median",p75_days=lambda s:s.quantile(.75)).reset_index())
        part["dimension"],part["group_value"],part["kpi_definition"]=dim,part[dim].astype(str),formula
        part["rank_median_desc"]=part["median_days"].rank(method="min",ascending=False).astype(int)
        part["sample_status"]=np.where(part["sample_size"]>=min_n,"meets_minimum","small_sample")
        tables.append(part.drop(columns=[dim]))
    return pd.concat(tables,ignore_index=True) if tables else pd.DataFrame()

def monthly_trend_table(monthly):
    """Ajoute variation absolue/relative et moyenne mobile 3 périodes sans imputation."""
    if monthly is None or monthly.empty: return pd.DataFrame()
    t=monthly.copy().sort_values("month_start").reset_index(drop=True)
    t["change_abs"]=t["request_count"].diff()
    t["change_pct"]=t["request_count"].pct_change()*100
    t["rolling_mean_3"]=t["request_count"].rolling(3,min_periods=3).mean()
    t["change_direction"]=np.select([t["change_abs"]>0,t["change_abs"]<0],["up","down"],default="flat")
    return t

def persistent_trend(trend_table, min_periods=4, required_same_direction=3):
    """Déclare une tendance persistante uniquement si plusieurs écarts successifs concordent."""
    if trend_table is None or len(trend_table)<min_periods:
        return {"status":"insufficient_periods","direction":"none","n_periods":0 if trend_table is None else len(trend_table)}
    diffs=trend_table["request_count"].diff().dropna()
    signs=np.sign(diffs.to_numpy())
    up=int((signs[-required_same_direction:]>0).sum())==required_same_direction
    down=int((signs[-required_same_direction:]<0).sum())==required_same_direction
    direction="up" if up else "down" if down else "none"
    return {"status":"persistent" if direction!="none" else "no_persistent_direction",
            "direction":direction,"n_periods":int(len(trend_table)),
            "start_period":str(trend_table.iloc[0]["month"]),"end_period":str(trend_table.iloc[-1]["month"]),
            "start_value":float(trend_table.iloc[0]["request_count"]),"end_value":float(trend_table.iloc[-1]["request_count"]),
            "change_abs":float(trend_table.iloc[-1]["request_count"]-trend_table.iloc[0]["request_count"]),
            "change_pct":float((trend_table.iloc[-1]["request_count"]/trend_table.iloc[0]["request_count"]-1)*100) if trend_table.iloc[0]["request_count"] else np.nan,
            "required_same_direction":required_same_direction}

def run_stl(series, period=12, min_months=MIN_STL_MONTHS):
    """Exécute STL uniquement sur une série mensuelle complète, régulière et sans NaN."""
    out={"status":"not_run","reason":None,"decomposition":None}
    if importlib.util.find_spec("statsmodels") is None:
        out.update(status="skipped",reason="statsmodels absent ; installer avec pip install statsmodels"); return out
    if series is None or len(series)<max(min_months,2*period):
        out.update(status="skipped",reason=f"historique insuffisant : minimum {max(min_months,2*period)} mois requis"); return out
    s=series.copy().sort_index()
    if not isinstance(s.index,pd.PeriodIndex):
        out.update(status="skipped",reason="index non mensuel PeriodIndex"); return out
    full=pd.period_range(s.index.min(),s.index.max(),freq="M")
    if not s.index.equals(full):
        out.update(status="skipped",reason="index mensuel irrégulier ou périodes manquantes ; aucune interpolation appliquée"); return out
    if s.isna().any() or not np.isfinite(s.to_numpy(dtype=float)).all():
        out.update(status="skipped",reason="valeurs manquantes/non finies ; aucune imputation appliquée"); return out
    from statsmodels.tsa.seasonal import STL
    fit=STL(s.astype(float),period=period,robust=True).fit()
    out.update(status="executed",reason=None,decomposition=pd.DataFrame({"observed":s,"trend":fit.trend,"seasonal":fit.seasonal,"resid":fit.resid}))
    return out

def stl_residual_signals(decomposition, threshold=MAD_THRESHOLD, min_n=MIN_REFERENCE_N):
    """Signale les résidus STL extrêmes contre les seuls résidus antérieurs (MAD robuste)."""
    columns=["period","observed","reference_median","mad","score","threshold","direction","status","reference_n","reference_period","method","dimension","group_value","rule_version"]
    if decomposition is None or decomposition.empty:return pd.DataFrame(columns=columns)
    rows=[]; residual=decomposition["resid"]
    for i in range(min_n,len(residual)):
        period=str(residual.index[i])
        r=detect_mad(residual.iloc[:i],float(residual.iloc[i]),threshold=threshold,min_n=min_n,period=period)
        r.update(method="STL_residual_MAD",dimension="all",group_value="all",rule_version=RULE_VERSION)
        rows.append(r)
    return pd.DataFrame(rows,columns=columns)

def compare_to_target(kpi, target=None, direction="lower_is_better", period=None, source=None):
    """Compare un KPI à une cible réellement fournie ; aucune cible n'est implicite."""
    if target is None:
        return {"kpi":kpi,"target":None,"period":period,"source":source,"status":"cible métier non configurée","gap":None}
    gap=float(kpi)-float(target)
    attained=(kpi<=target) if direction=="lower_is_better" else (kpi>=target)
    return {"kpi":float(kpi),"target":float(target),"period":period,"source":source,"direction":direction,
            "gap":gap,"status":"atteinte" if attained else "dépassée"}

group_comparisons=compare_group_delays(df,["borough","complaint_type","agency"])
partial_months=set(sample_metadata.get("partial_months", [])) if sample_metadata is not None else set()
trend_monthly_volume=(monthly_volume.loc[~monthly_volume["month"].isin(partial_months)].copy() if monthly_volume is not None else None)
if monthly_volume is not None and len(partial_months): print("Mois partiels exclus des tendances :", sorted(partial_months))
trend_analysis=monthly_trend_table(trend_monthly_volume)
volume_trend=temporal_trend(trend_monthly_volume,"request_count","monthly_request_volume")
persistent_volume_trend=persistent_trend(trend_analysis)
trend_monthly_delay=(monthly_delay.loc[~monthly_delay["month"].isin(partial_months)].copy() if monthly_delay is not None else None)
delay_trend=temporal_trend(trend_monthly_delay,"median_days","monthly_median_resolution_days")
trend_results=pd.DataFrame([volume_trend,delay_trend])
monthly_series=(trend_monthly_volume.set_index("month")["request_count"] if trend_monthly_volume is not None else None)
if monthly_series is not None:
    monthly_series.index=pd.PeriodIndex(monthly_series.index,freq="M")
stl_result=run_stl(monthly_series,period=12)
stl_decomposition=stl_result["decomposition"]
stl_residual_comparison=stl_residual_signals(stl_decomposition)
median_resolution=float(delay_distribution.loc[0,"median_days"]) if pd.notna(delay_distribution.loc[0,"median_days"]) else np.nan
target_check=compare_to_target(median_resolution,None,direction="lower_is_better",period="période du fichier",source=None)

def scan_monthly_anomalies(table, value_col="request_count", methods=("zscore","mad")):
    """Compare chaque mois à ses seules périodes antérieures et retourne preuves normal/anomalie."""
    rows=[]
    cols=["observed","reference_mean","reference_std","score","threshold","direction","period","reference_n","reference_period","status","method","dimension","group_value","rule_version"]
    if table is None or table.empty:return pd.DataFrame(columns=cols)
    ordered=table.sort_values("month_start").reset_index(drop=True)
    for i in range(MIN_REFERENCE_N,len(ordered)):
        ref=ordered.loc[:i-1,value_col]
        period=str(ordered.loc[i,"month"]); observed=float(ordered.loc[i,value_col])
        for method in methods:
            r=detect_zscore(ref,observed,period=period) if method=="zscore" else detect_mad(ref,observed,period=period)
            r["reference_period"]=f"{counts.loc[0,'month']}–{counts.loc[i-1,'month']}"
            r.update(method=method,dimension="all",group_value="all",rule_version=RULE_VERSION)
            rows.append(r)
    return pd.DataFrame(rows,columns=cols)

anomaly_comparison=scan_monthly_anomalies(trend_monthly_volume)
def scan_group_month_anomalies(frame, dimension):
    """Scanne les volumes mensuels de chaque groupe, avec historique antérieur suffisant."""
    if dimension not in frame or "month" not in frame:return pd.DataFrame()
    rows=[]
    for group_value,part in frame.dropna(subset=[dimension,"month"]).groupby(dimension):
        counts=part.groupby("month").size().rename("request_count").reset_index()
        counts["month_start"]=pd.PeriodIndex(counts["month"],freq="M").to_timestamp()
        result=scan_monthly_anomalies(counts)
        if len(result):
            result["dimension"],result["group_value"]=dimension,str(group_value)
            rows.append(result)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

group_anomaly_parts=[scan_group_month_anomalies(df,c) for c in ["complaint_type","borough","agency"] if c in df]
group_anomaly_parts=[x for x in group_anomaly_parts if len(x)]
if group_anomaly_parts:
    anomaly_comparison=pd.concat([anomaly_comparison,*group_anomaly_parts],ignore_index=True)
advanced_anomalies=anomaly_comparison.loc[anomaly_comparison["status"]=="anomaly"].copy() if len(anomaly_comparison) else anomaly_comparison.copy()
if len(stl_residual_comparison):
    stl_residual_anomalies=stl_residual_comparison.loc[stl_residual_comparison["status"]=="anomaly"].copy()
    if len(stl_residual_anomalies):
        advanced_anomalies=pd.concat([advanced_anomalies,stl_residual_anomalies],ignore_index=True,sort=False)
print("Comparaisons de groupes :",len(group_comparisons),"| mois pour tendance :",len(trend_analysis),
      "| STL :",stl_result["status"],stl_result["reason"],"| cible :",target_check["status"])
display(group_comparisons.head(20)); display(trend_analysis); display(anomaly_comparison); display(advanced_anomalies)

Mois partiels exclus des tendances : ['2026-10']


Comparaisons de groupes : 143 | mois pour tendance : 4 | STL : skipped historique insuffisant : minimum 24 mois requis | cible : cible métier non configurée


,sample_size,mean_days,median_days,p75_days,dimension,group_value,kpi_definition,rank_median_desc,sample_status
0,1828,3.503330,0.147436,2.022344,borough,BRONX,median(resolution_days) groupé par catégorie ; afficher sample_size,6,meets_minimum
1,3080,4.732281,0.228484,3.864528,borough,BROOKLYN,median(resolution_days) groupé par catégorie ; afficher sample_size,3,meets_minimum
2,1712,4.748242,0.170666,3.382648,borough,MANHATTAN,median(resolution_days) groupé par catégorie ; afficher sample_size,4,meets_minimum
3,2274,3.319092,0.165943,1.115145,borough,QUEENS,median(resolution_days) groupé par catégorie ; afficher sample_size,5,meets_minimum
4,386,3.624707,0.332269,2.294789,borough,STATEN ISLAND,median(resolution_days) groupé par catégorie ; afficher sample_size,2,meets_minimum
5,5,15.333641,2.628472,23.173009,borough,Unspecified,median(resolution_days) groupé par catégorie ; afficher sample_size,1,small_sample
6,52,18.156573,17.245417,29.317095,complaint_type,APPLIANCE,median(resolution_days) groupé par catégorie ; afficher sample_size,10,meets_minimum
7,7,4.485117,2.794664,6.616997,complaint_type,Abandoned Bike,median(resolution_days) groupé par catégorie ; afficher sample_size,61,small_sample
8,203,0.160093,0.079630,0.191181,complaint_type,Abandoned Vehicle,median(resolution_days) groupé par catégorie ; afficher sample_size,104,meets_minimum
9,19,3.904388,1.884676,5.345660,complaint_type,Air Quality,median(resolution_days) groupé par catégorie ; afficher sample_size,72,small_sample


,month,request_count,sample_count,count_basis,month_start,change_abs,change_pct,rolling_mean_3,change_direction
0,2026-06,334834,2242,official monthly aggregate; sample used for dimensions/delays,2026-06-01,NaN,NaN,NaN,flat
1,2026-07,343133,2298,official monthly aggregate; sample used for dimensions/delays,2026-07-01,8299.0,2.478542,NaN,up
2,2026-08,328918,2203,official monthly aggregate; sample used for dimensions/delays,2026-08-01,-14215.0,-4.142709,335628.333333,down
3,2026-09,323627,2167,official monthly aggregate; sample used for dimensions/delays,2026-09-01,-5291.0,-1.608608,331892.666667,down


,observed,reference_mean,reference_std,score,threshold,direction,period,reference_n,reference_period,status,method,dimension,group_value,rule_version


,observed,reference_mean,reference_std,score,threshold,direction,period,reference_n,reference_period,status,method,dimension,group_value,rule_version


## 4. Anomalies de volume et délais relativement élevés

Pour une série mensuelle suffisamment longue, la référence utilise le score robuste `0,6745 × (volume − médiane) / MAD`, avec seuil absolu de **3,5**. Il faut au moins six mois. Si le MAD est nul (série très stable), un repli explicite signale les écarts d’au moins 20 % de la médiane. Ce seuil de repli sert à détecter des écarts à investiguer ; ce n’est pas un seuil de qualité métier. Le calcul s’applique au total et, si disponibles, aux catégories/arrondissements observés sur plusieurs mois.

Pour les délais, les groupes sont comparés à la distribution des médianes de leurs pairs ; seuls les groupes ayant au moins **30** délais valides sont classés. Le 75e percentile indique des délais relativement élevés, sans SLA ni jugement de non-performance. Vérifier les anomalies contre la couverture, les doublons et les changements de collecte ; ces causes ne peuvent pas être établies par ce seul extrait.

In [6]:
MIN_MONTHS_ANOMALY = 6
ROBUST_Z_THRESHOLD = 3.5
ZERO_MAD_RELATIVE_THRESHOLD = 0.20
MIN_DELAY_GROUP_N = 30

def robust_month_anomalies(frame, group_column=None, group_label="all"):
    if "month" not in frame.columns or frame["month"].notna().sum() == 0:
        return []
    subset = frame.dropna(subset=["month"])
    group_iter = [(group_label, subset)] if group_column is None else [
        (str(value), grp) for value, grp in subset.dropna(subset=[group_column]).groupby(group_column)
    ]
    found = []
    for group_value, grp in group_iter:
        counts = grp.groupby("month").size().rename("volume").reset_index()
        counts["month_start"] = pd.PeriodIndex(counts["month"], freq="M").to_timestamp()
        counts = counts.sort_values("month_start")
        if len(counts) < MIN_MONTHS_ANOMALY:
            continue
        vals = counts["volume"].astype(float)
        median = float(vals.median())
        mad = float((vals - median).abs().median())
        if mad > 0:
            scores = 0.6745 * (vals - median) / mad
            mask = scores.abs() >= ROBUST_Z_THRESHOLD
            method = "robust_z_mad"
        else:
            scores = pd.Series(np.nan, index=vals.index)
            mask = (vals - median).abs() >= max(1.0, median * ZERO_MAD_RELATIVE_THRESHOLD)
            method = "fallback_relative_deviation_mad_zero"
        for idx in counts.index[mask]:
            found.append({"dimension": group_label if group_column is None else group_column,
                          "group": group_value, "period": counts.loc[idx, "month"],
                          "volume": int(counts.loc[idx, "volume"]), "baseline_median": median,
                          "mad": mad, "score": float(scores.loc[idx]) if pd.notna(scores.loc[idx]) else np.nan,
                          "method": method, "threshold": ROBUST_Z_THRESHOLD if mad > 0 else ZERO_MAD_RELATIVE_THRESHOLD,
                          "n_months": int(len(counts))})
    return found

anomaly_records = robust_month_anomalies(df)
for col in ["complaint_type", "borough"]:
    if col in df.columns:
        anomaly_records.extend(robust_month_anomalies(df, col, col))
anomaly_columns = ["dimension", "group", "period", "volume", "baseline_median", "mad", "score", "method", "threshold", "n_months"]
anomalies = pd.DataFrame(anomaly_records, columns=anomaly_columns)
print("Anomalies détectées :", len(anomalies), "| règle : MAD robuste seuil", ROBUST_Z_THRESHOLD,
      "| minimum", MIN_MONTHS_ANOMALY, "mois")
display(anomalies.sort_values("score", key=lambda s: s.abs(), ascending=False) if len(anomalies) else anomalies)

def delay_groups(frame, group_col):
    if group_col not in frame.columns or "resolution_days" not in frame.columns:
        return None
    pop = frame.loc[valid_delay_mask & frame[group_col].notna()]
    table = (pop.groupby(group_col)["resolution_days"].agg(median_days="median", sample_size="size")
             .reset_index())
    return table.loc[table["sample_size"] >= MIN_DELAY_GROUP_N].sort_values("median_days", ascending=False)

delay_by_type = delay_groups(df, "complaint_type")
delay_by_borough = delay_groups(df, "borough")
high_delay_groups = []
for dim, table in [("complaint_type", delay_by_type), ("borough", delay_by_borough)]:
    if table is None:
        print(f"Délai par {dim} indisponible : colonne ou délai absent.")
        continue
    if len(table):
        cutoff = float(table["median_days"].quantile(.75))
        for _, row in table.loc[table["median_days"] >= cutoff].iterrows():
            high_delay_groups.append({"dimension": dim, "group": str(row[dim]),
                                      "median_days": float(row["median_days"]),
                                      "sample_size": int(row["sample_size"]), "peer_p75_days": cutoff,
                                      "minimum_group_n": MIN_DELAY_GROUP_N})
        display(table.head(10))
high_delay = pd.DataFrame(high_delay_groups, columns=["dimension", "group", "median_days", "sample_size", "peer_p75_days", "minimum_group_n"])
print("Groupes aux médianes dans le quart supérieur de leurs groupes admissibles (comparaison descriptive, aucun SLA) :")
display(high_delay)
print("À vérifier : un pic peut refléter un mois partiel, un changement de collecte ou des répétitions ; le CSV seul ne tranche pas la cause.")

Anomalies détectées : 0 | règle : MAD robuste seuil 3.5 | minimum 6 mois


,dimension,group,period,volume,baseline_median,mad,score,method,threshold,n_months


,complaint_type,median_days,sample_size
0,APPLIANCE,17.245417,52
98,Sidewalk Condition,11.916233,80
66,Maintenance or Facility,10.728501,84
21,DOOR/WINDOW,10.241389,88
111,UNSANITARY CONDITION,10.171447,299
118,WATER LEAK,10.155584,98
91,Rodent,9.970764,64
103,Street Light Condition,9.777083,97
15,Building/Use,9.505845,63
42,GENERAL,8.353831,91


,borough,median_days,sample_size
4,STATEN ISLAND,0.332269,386
1,BROOKLYN,0.228484,3080
2,MANHATTAN,0.170666,1712
3,QUEENS,0.165943,2274
0,BRONX,0.147436,1828


Groupes aux médianes dans le quart supérieur de leurs groupes admissibles (comparaison descriptive, aucun SLA) :


,dimension,group,median_days,sample_size,peer_p75_days,minimum_group_n
0,complaint_type,APPLIANCE,17.245417,52,5.977986,30
1,complaint_type,Sidewalk Condition,11.916233,80,5.977986,30
2,complaint_type,Maintenance or Facility,10.728501,84,5.977986,30
3,complaint_type,DOOR/WINDOW,10.241389,88,5.977986,30
4,complaint_type,UNSANITARY CONDITION,10.171447,299,5.977986,30
5,complaint_type,WATER LEAK,10.155584,98,5.977986,30
6,complaint_type,Rodent,9.970764,64,5.977986,30
7,complaint_type,Street Light Condition,9.777083,97,5.977986,30
8,complaint_type,Building/Use,9.505845,63,5.977986,30
9,complaint_type,GENERAL,8.353831,91,5.977986,30


À vérifier : un pic peut refléter un mois partiel, un changement de collecte ou des répétitions ; le CSV seul ne tranche pas la cause.


## 5. Génération déterministe des fiches d’insights

Une fiche sépare le fait observé, son interprétation possible et l’action à envisager. Elle est générée uniquement à partir des tendances, anomalies et médianes calculées ci-dessus. La confiance est une appréciation de la quantité de données (périodes/effectifs), pas une probabilité. Une recommandation invite à vérifier ou investiguer ; elle n’attribue pas de cause.

In [7]:
def confidence_for(n, periods=0):
    if periods >= 12 and n >= 100: return "medium"
    if periods >= 6 and n >= 30: return "medium"
    return "low"

def make_insight(insight_id, insight_type, period, dimension, observation, evidence,
                 interpretation, recommendation, confidence, limitations):
    return {"insight_id": insight_id, "insight_type": insight_type, "period": period,
            "dimension": dimension, "observation": observation, "evidence": evidence,
            "interpretation": interpretation, "recommendation": recommendation,
            "confidence": confidence, "limitations": limitations}

insights = []
if volume_trend["direction"] in ["up", "down"]:
    insights.append(make_insight(
        f"trend-volume-{len(insights)+1:03d}", "tendance", "période observée", "volume total",
        f"Le volume mensuel suit une tendance {('à la hausse' if volume_trend['direction']=='up' else 'à la baisse')}.",
        {k: (None if pd.isna(v) else v) for k,v in volume_trend.items()},
        "La pente résume la période observée ; elle ne permet pas d’identifier une cause.",
        "Vérifier les mois de début et de fin, puis comparer avec les événements de service et la couverture de collecte.",
        confidence_for(volume_trend["n_periods"], volume_trend["n_periods"]),
        "Tendance descriptive linéaire ; série courte ou mois partiel possibles."))
for _, row in anomalies.iterrows():
    evidence = {k: (None if pd.isna(v) else v.item() if isinstance(v, np.generic) else v) for k,v in row.to_dict().items()}
    insights.append(make_insight(
        f"anomaly-{len(insights)+1:03d}", "anomalie", str(row["period"]), f"{row['dimension']}={row['group']}",
        f"Volume atypique détecté : {int(row['volume'])} demandes, référence médiane {row['baseline_median']:.1f}.",
        evidence, "Écart statistique à examiner ; il ne prouve pas une anomalie opérationnelle.",
        "Contrôler l’exhaustivité du mois, les doublons et les changements de collecte avant d’alerter l’équipe.",
        confidence_for(int(row["volume"]), int(row["n_months"])),
        "Score robuste exploratoire ; cause non identifiable dans ces champs."))
for item in high_delay_groups:
    insights.append(make_insight(
        f"delay-{len(insights)+1:03d}", "délai élevé", "période couverte", f"{item['dimension']}={item['group']}",
        f"Médiane de clôture {item['median_days']:.1f} jours parmi {item['sample_size']} demandes valides.",
        item, "Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.",
        "Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.",
        confidence_for(item["sample_size"]),
        "Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."))

insight_columns = ["insight_id", "insight_type", "period", "dimension", "observation", "evidence",
                   "interpretation", "recommendation", "confidence", "limitations"]
insights_df = pd.DataFrame(insights, columns=insight_columns)
if len(insights_df):
    insights_df["evidence"] = insights_df["evidence"].apply(lambda x: json.dumps(x, ensure_ascii=False, default=str))
display(insights_df)
print("Nombre de fiches construites à partir des résultats mesurés :", len(insights_df))

,insight_id,insight_type,period,dimension,observation,evidence,interpretation,recommendation,confidence,limitations
0,trend-volume-001,tendance,période observée,volume total,Le volume mensuel suit une tendance à la baisse.,"{""metric"": ""monthly_request_volume"", ""n_periods"": 4, ""direction"": ""down"", ""slope_per_month"": -4783.600000000055, ""fi...",La pente résume la période observée ; elle ne permet pas d’identifier une cause.,"Vérifier les mois de début et de fin, puis comparer avec les événements de service et la couverture de collecte.",low,Tendance descriptive linéaire ; série courte ou mois partiel possibles.
1,delay-002,délai élevé,période couverte,complaint_type=APPLIANCE,Médiane de clôture 17.2 jours parmi 52 demandes valides.,"{""dimension"": ""complaint_type"", ""group"": ""APPLIANCE"", ""median_days"": 17.245416666666667, ""sample_size"": 52, ""peer_p7...",Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.,Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.,low,"Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."
2,delay-003,délai élevé,période couverte,complaint_type=Sidewalk Condition,Médiane de clôture 11.9 jours parmi 80 demandes valides.,"{""dimension"": ""complaint_type"", ""group"": ""Sidewalk Condition"", ""median_days"": 11.91623263888889, ""sample_size"": 80, ...",Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.,Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.,low,"Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."
3,delay-004,délai élevé,période couverte,complaint_type=Maintenance or Facility,Médiane de clôture 10.7 jours parmi 84 demandes valides.,"{""dimension"": ""complaint_type"", ""group"": ""Maintenance or Facility"", ""median_days"": 10.728501157407408, ""sample_size""...",Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.,Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.,low,"Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."
4,delay-005,délai élevé,période couverte,complaint_type=DOOR/WINDOW,Médiane de clôture 10.2 jours parmi 88 demandes valides.,"{""dimension"": ""complaint_type"", ""group"": ""DOOR/WINDOW"", ""median_days"": 10.241388888888888, ""sample_size"": 88, ""peer_...",Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.,Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.,low,"Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."
5,delay-006,délai élevé,période couverte,complaint_type=UNSANITARY CONDITION,Médiane de clôture 10.2 jours parmi 299 demandes valides.,"{""dimension"": ""complaint_type"", ""group"": ""UNSANITARY CONDITION"", ""median_days"": 10.17144675925926, ""sample_size"": 29...",Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.,Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.,low,"Comparaison descriptive ; différences de composition des demandes possibles, aucune causalité établie."
6,delay-007,délai élevé,période couverte,complaint_type=WATER LEAK,Médiane de clôture 10.2 jours parmi 98 demandes valides.,"{""dimension"": ""complaint_type"", ""group"": ""WATER LEAK"", ""median_days"": 10.15558449074074, ""sample_size"": 98, ""peer_p7...",Cette médiane se situe dans le quart supérieur des groupes comparables admissibles.,Examiner un échantillon de dossiers et les étapes de traitement pour comprendre les délais ; aucun SLA n’est fourni.,low,"Comparaison descriptive ; diffé

Nombre de fiches construites à partir des résultats mesurés : 16


## Insights enrichis, preuves et auditabilité

Les fiches ci-dessous combinent uniquement résultats mesurés et règles versionnées. Chaque fiche contient KPI/définition, valeurs de référence, score, seuil, période, règle déclenchée, limites, priorité calculée par écart relatif (≥50 %), effectif (≥100) et signal de doublons, ainsi qu'un état de qualité. La priorité (1–3) est un rang de tri déterministe, pas une probabilité. La source d'un insight reste retrouvable via `evidence_id` dans le tableau de preuves agrégées.

In [8]:
def build_auditable_insights(anomaly_rows, group_delays, trend_state, quality_duplicates, target_state):
    """Construit des insights à règles fixes et un registre détaillé de preuves."""
    records=[]; evidence_rows=[]
    volume_def=(catalog.loc[catalog["kpi_name"]=="monthly_request_volume","business_definition"].iloc[0]
                if catalog is not None and (catalog["kpi_name"]=="monthly_request_volume").any()
                else "Définition KPI absente du catalogue ; volume agrégé par lignes datées")
    def add(kind,period,dimension,group,observation,evidence,method,threshold,reference_period,interpretation,recommendation,n,quality,limitations,rule):
        rel=abs(float(evidence.get("relative_deviation",0) or 0))
        quality_text=str(quality)
        quality_risk=("doublons_raw_période=" in quality_text and not quality_text.endswith("=0"))
        # Priorité de tri (1–3), pas une probabilité : ampleur ≥50 %, effectif ≥100, signal qualité.
        priority=min(3,1+int(rel>=.5)+int(n>=100)+int(quality_risk))
        iid=f"BI-{len(records)+1:04d}"
        eid=f"EV-{len(evidence_rows)+1:04d}"
        evidence_full={**evidence,"evidence_id":eid,"rule_id":rule,"rule_version":RULE_VERSION,
                      "kpi_definition":volume_def if kind in ("hausse inhabituelle","baisse inhabituelle","tendance persistante","données insuffisantes") else "Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI."}
        records.append({"insight_id":iid,"insight_type":kind,"period":period,"dimension":dimension,"group_value":group,
                        "observation":observation,"evidence":json.dumps(evidence_full,ensure_ascii=False,default=str),"method":method,
                        "threshold":threshold,"reference_period":reference_period,"interpretation":interpretation,
                        "recommendation":recommendation,"priority":priority,"confidence":"medium" if n>=100 else "low",
                        "limitations":limitations,"data_quality_status":quality,"evidence_id":eid,"rule_id":rule})
        evidence_rows.append({"evidence_id":eid,"insight_id":iid,"period":period,"dimension":dimension,"group_value":group,
                              "observed":evidence.get("observed"),"reference":evidence.get("reference"),"score":evidence.get("score"),
                              "threshold":threshold,"method":method,"reference_period":reference_period,
                              "kpi_definition":evidence_full["kpi_definition"],"rule_id":rule,"rule_version":RULE_VERSION,
                              "aggregated_values":json.dumps(evidence.get("aggregated_values",{}),ensure_ascii=False,default=str),
                              "limitations":limitations})
    if trend_state["status"]=="persistent":
        add("tendance persistante",f"{trend_state['start_period']}–{trend_state['end_period']}","volume mensuel","all",
            f"Tendance persistante {trend_state['direction']} sur les dernières périodes.",
            {"observed":trend_state["end_value"],"reference":trend_state["start_value"],"relative_deviation":trend_state["change_pct"],"aggregated_values":trend_state},
            "variations mensuelles successives",trend_state["required_same_direction"],trend_state["start_period"],
            "Plusieurs variations successives vont dans le même sens ; aucune cause n'est établie.",
            "Vérifier la couverture des périodes et examiner les catégories/zones qui contribuent au changement.",len(trend_analysis),
            raw_quality_status,"Tendance descriptive ; mois incomplets et saisonnalité peuvent influencer la série.","RULE_TREND_PERSISTENT")
    elif trend_state["status"]=="insufficient_periods":
        add("données insuffisantes",None,"volume mensuel","all",
            f"Pas de tendance persistante conclue : {trend_state['status']}.",
            {"observed":None,"reference":None,"aggregated_values":{"period_count":trend_state.get("n_periods",0)}},
            "contrôle de longueur/persistance",4,None,"Les données ne satisfont pas la règle de persistance.",
            "Charger davantage de périodes mensuelles complètes avant d'interpréter une tendance.",int(trend_state.get("n_periods",0)),
            raw_quality_status,"Historique trop court ou changements mensuels non persistants.","RULE_INSUFFICIENT_HISTORY")
    for _,r in anomaly_rows.iterrows():
        direction=r["direction"]
        kind="hausse inhabituelle" if direction=="positive" else "baisse inhabituelle"
        period=r.get("period"); duplicate_n=0
        if len(quality_duplicates) and "month" in quality_duplicates:
            matched=quality_duplicates.loc[quality_duplicates["month"].astype(str)==str(period),"raw_duplicate_rows"]
            duplicate_n=int(matched.sum()) if len(matched) else 0
        qstatus=f"anomalie_statistique; doublons_raw_période={duplicate_n}"
        quality_note="Doublons observés dans le brut sur cette période ; pic potentiellement lié à la qualité, vérifier les clés." if duplicate_n else "Aucun doublon par clé observé dans le contrôle disponible ; autres changements de collecte non vérifiés."
        add(kind,period,r.get("dimension","all"),r.get("group_value","all"),
            f"Valeur {r['observed']:.1f}, score {r['score']:.2f} ({direction}).",
            {"observed":r["observed"],"reference":r.get("reference_mean",r.get("reference_median")),
             "score":r["score"],"relative_deviation":(r["observed"]-r.get("reference_mean",r.get("reference_median")))/max(abs(r.get("reference_mean",r.get("reference_median"))),1e-9),
             "aggregated_values":{"reference_n":int(r["reference_n"]),"reference_std":r.get("reference_std"),"mad":r.get("mad"),"raw_duplicate_rows":duplicate_n}},
            r["method"],r["threshold"],r["reference_period"],
            "Écart statistique à examiner ; il ne prouve ni erreur ni cause opérationnelle.",
            "Vérifier doublons, catégories, zones et couverture de collecte avant de conclure.",int(r["reference_n"]),qstatus,
            quality_note,"RULE_VOLUME_ANOMALY")
    if not group_delays.empty:
        for dim,part in group_delays[group_delays["sample_status"]=="meets_minimum"].groupby("dimension"):
            cutoff=float(part["median_days"].quantile(.75))
            for _,r in part.loc[part["median_days"]>=cutoff].iterrows():
                add("délai relativement élevé",None,dim,r["group_value"],
                    f"Médiane {r['median_days']:.3f} jours (n={int(r['sample_size'])}), quart supérieur comparatif.",
                    {"observed":float(r["median_days"]),"reference":cutoff,"relative_deviation":(float(r["median_days"])-cutoff)/max(abs(cutoff),1e-9),
                     "aggregated_values":{"sample_size":int(r["sample_size"]),"mean_days":float(r["mean_days"]),"p75_days":float(r["p75_days"])}},
                    "médiane de groupe comparée au P75 des groupes admissibles",None,"groupes de même dimension",
                    "Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",
                    "Examiner des dossiers et le parcours de traitement ; valider les différences de composition.",int(r["sample_size"]),
                    raw_quality_status,"Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.","RULE_GROUP_DELAY_P75")
    if target_state["status"]=="cible métier non configurée":
        # Conserver le statut d'absence de cible dans le registre, sans créer un constat de dépassement.
        pass
    return pd.DataFrame(records),pd.DataFrame(evidence_rows)

rule_insights, evidence_registry=build_auditable_insights(advanced_anomalies,group_comparisons,
                                                         persistent_volume_trend,raw_duplicate_by_month,target_check)
print("Insights auditables :",len(rule_insights),"| preuves traçables :",len(evidence_registry),"| version :",RULE_VERSION)
display(rule_insights); display(evidence_registry.head(10))

Insights auditables : 18 | preuves traçables : 18 | version : biflow-bi-analyst-rules-1.0


,insight_id,insight_type,period,dimension,group_value,observation,evidence,method,threshold,reference_period,interpretation,recommendation,priority,confidence,limitations,data_quality_status,evidence_id,rule_id
0,BI-0001,délai relativement élevé,None,agency,DCWP,"Médiane 4.620 jours (n=55), quart supérieur comparatif.","{""observed"": 4.619965277777778, ""reference"": 4.391585648148149, ""relative_deviation"": 0.05200391109892919, ""aggregat...",médiane de groupe comparée au P75 des groupes admissibles,None,groupes de même dimension,"Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",Examiner des dossiers et le parcours de traitement ; valider les différences de composition.,1,low,Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.,contrôle_clé_raw_effectué,EV-0001,RULE_GROUP_DELAY_P75
1,BI-0002,délai relativement élevé,None,agency,DOHMH,"Médiane 10.878 jours (n=166), quart supérieur comparatif.","{""observed"": 10.87788773148148, ""reference"": 4.391585648148149, ""relative_deviation"": 1.4769840788755846, ""aggregate...",médiane de groupe comparée au P75 des groupes admissibles,None,groupes de même dimension,"Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",Examiner des dossiers et le parcours de traitement ; valider les différences de composition.,3,medium,Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.,contrôle_clé_raw_effectué,EV-0002,RULE_GROUP_DELAY_P75
2,BI-0003,délai relativement élevé,None,agency,HPD,"Médiane 7.401 jours (n=1252), quart supérieur comparatif.","{""observed"": 7.401331018518518, ""reference"": 4.391585648148149, ""relative_deviation"": 0.6853436575100212, ""aggregate...",médiane de groupe comparée au P75 des groupes admissibles,None,groupes de même dimension,"Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",Examiner des dossiers et le parcours de traitement ; valider les différences de composition.,3,medium,Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.,contrôle_clé_raw_effectué,EV-0003,RULE_GROUP_DELAY_P75
3,BI-0004,délai relativement élevé,None,borough,BROOKLYN,"Médiane 0.228 jours (n=3080), quart supérieur comparatif.","{""observed"": 0.22848379629629623, ""reference"": 0.22848379629629623, ""relative_deviation"": 0.0, ""aggregated_values"": ...",médiane de groupe comparée au P75 des groupes admissibles,None,groupes de même dimension,"Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",Examiner des dossiers et le parcours de traitement ; valider les différences de composition.,2,medium,Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.,contrôle_clé_raw_effectué,EV-0004,RULE_GROUP_DELAY_P75
4,BI-0005,délai relativement élevé,None,borough,STATEN ISLAND,"Médiane 0.332 jours (n=386), quart supérieur comparatif.","{""observed"": 0.33226851851851846, ""reference"": 0.22848379629629623, ""relative_deviation"": 0.4542323083936986, ""aggre...",médiane de groupe comparée au P75 des groupes admissibles,None,groupes de même dimension,"Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",Examiner des dossiers et le parcours de traitement ; valider les différences de composition.,2,medium,Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.,contrôle_clé_raw_effectué,EV-0005,RULE_GROUP_DELAY_P75
5,BI-0006,délai relativement élevé,None,complaint_type,APPLIANCE,"Médiane 17.245 jours (n=52), quart supérieur comparatif.","{""observed"": 17.245416666666667, ""reference"": 5.977986111111111, ""relative_deviation"": 1.884820463970819, ""aggregate...",médiane de groupe comparée au P75 des groupes admissibles,None,groupes de même dimension,"Délai relativement élevé parmi les groupes comparables admissibles, pas un dépassement SLA.",Examiner des dossiers et le parcours de traitement ; valider les différen

,evidence_id,insight_id,period,dimension,group_value,observed,reference,score,threshold,method,reference_period,kpi_definition,rule_id,rule_version,aggregated_values,limitations
0,EV-0001,BI-0001,None,agency,DCWP,4.619965,4.391586,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 55, ""mean_days"": 17.742987163299663, ""p75_days"": 32.60554398148148}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
1,EV-0002,BI-0002,None,agency,DOHMH,10.877888,4.391586,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 166, ""mean_days"": 22.38732638888889, ""p75_days"": 39.055908564814814}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
2,EV-0003,BI-0003,None,agency,HPD,7.401331,4.391586,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 1252, ""mean_days"": 11.720518633149922, ""p75_days"": 16.934505208333334}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
3,EV-0004,BI-0004,None,borough,BROOKLYN,0.228484,0.228484,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 3080, ""mean_days"": 4.732281287578163, ""p75_days"": 3.8645283564814816}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
4,EV-0005,BI-0005,None,borough,STATEN ISLAND,0.332269,0.228484,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 386, ""mean_days"": 3.6247074997601225, ""p75_days"": 2.2947887731481478}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
5,EV-0006,BI-0006,None,complaint_type,APPLIANCE,17.245417,5.977986,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 52, ""mean_days"": 18.15657340633903, ""p75_days"": 29.31709490740741}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
6,EV-0007,BI-0007,None,complaint_type,Building/Use,9.505845,5.977986,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 63, ""mean_days"": 23.53951682833627, ""p75_days"": 43.58122685185185}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
7,EV-0008,BI-0008,None,complaint_type,DOOR/WINDOW,10.241389,5.977986,None,None,médiane de groupe comparée au P75 des groupes admissibles,groupes de même dimension,Médiane de resolution_days sur demandes clôturées exploitables ; définition selon catalogue KPI.,RULE_GROUP_DELAY_P75,biflow-bi-analyst-rules-1.0,"{""sample_size"": 88, ""mean_days"": 12.978880076809764, ""p75_days"": 16.966880787037038}",Comparaison descriptive ; aucun seuil officiel de délai n'est configuré.
8,EV-0009,BI-0009,None,complaint_type,ELECTRIC,7.081030,5.977986,None,None,médiane de gro

## 6. Synthèse destinée au responsable de service

La synthèse reformule uniquement les résultats disponibles en langage courant. Les recommandations sont limitées aux investigations que les données justifient. Si la série est trop courte ou qu’aucune anomalie ne franchit le seuil, le rapport le dit explicitement.

In [9]:
def fmt_num(value, decimals=1):
    return "non calculable" if pd.isna(value) else f"{value:,.{decimals}f}".replace(",", " ")

report_lines = ["# Rapport Analyste BI — NYC 311", "",
                f"Échantillon analysé : {len(df):,} lignes."]
if "created_date" in df:
    report_lines.append(f"Période disponible : {df['created_date'].min()} à {df['created_date'].max()}.")
report_lines += ["", "## Tendances", ""]
if volume_trend["direction"] == "insufficient_data":
    report_lines.append(f"Pas assez de mois observés pour conclure sur une tendance de volume ({volume_trend['n_periods']} mois ; minimum 3).")
else:
    report_lines.append(f"Volume : tendance {('à la hausse' if volume_trend['direction']=='up' else 'à la baisse' if volume_trend['direction']=='down' else 'stable')}, pente descriptive {fmt_num(volume_trend['slope_per_month'])} demandes/mois sur {volume_trend['n_periods']} mois.")
if delay_trend["direction"] == "insufficient_data":
    report_lines.append(f"Délais mensuels : pas assez de périodes valides pour comparer ({delay_trend['n_periods']} mois).")
else:
    report_lines.append(f"Médiane des délais : tendance {delay_trend['direction']} sur {delay_trend['n_periods']} mois.")
report_lines += ["", "## Anomalies à examiner", ""]
if anomalies.empty:
    report_lines.append(f"Détection mensuelle non exécutée : {len(monthly_volume) if monthly_volume is not None else 0} mois disponibles, alors qu’il faut au moins 7 pour comparer une période à son historique. Aucune conclusion d’absence d’anomalie n’est possible.")
else:
    for _, r in anomalies.head(5).iterrows():
        report_lines.append(f"- {r['period']} — {r['dimension']} {r['group']} : {int(r['volume'])} demandes, médiane de référence {fmt_num(r['baseline_median'])}.")
report_lines += ["", "## Délais relativement élevés", ""]
if high_delay.empty:
    report_lines.append(f"Aucun groupe avec au moins {MIN_DELAY_GROUP_N} délais valides n’a été identifié dans le quart supérieur, ou les données sont insuffisantes.")
else:
    for item in high_delay_groups[:5]:
        report_lines.append(f"- {item['dimension']} {item['group']} : médiane {fmt_num(item['median_days'])} jours (n={item['sample_size']}, seuil comparatif P75={fmt_num(item['peer_p75_days'])} jours). Ce n’est pas un constat de non-respect d’un SLA.")
report_lines += ["", "## Recommandations prioritaires", ""]
recommendations = []
if not anomalies.empty:
    recommendations.append("Vérifier la couverture, les doublons et les changements de collecte pour les périodes signalées avant toute escalade.")
if not high_delay.empty:
    recommendations.append("Examiner un échantillon de dossiers dans les groupes aux délais relativement élevés afin d’identifier les étapes à investiguer.")
if volume_trend["direction"] != "insufficient_data" or delay_trend["direction"] != "insufficient_data":
    recommendations.append("Confirmer que les mois comparés sont complets et comparables, surtout aux bornes de l’extrait.")
if recommendations:
    report_lines.extend([f"{i}. {r}" for i,r in enumerate(recommendations[:3],1)])
else:
    report_lines.append("Les données actuelles ne justifient pas de recommandation opérationnelle spécifique ; obtenir une période plus longue et vérifier la couverture.")
report_lines += ["", "## Limites et vérifications", "",
                 "- Les résultats décrivent l’extrait fourni, pas nécessairement toutes les demandes de New York.",
                 "- Les demandes sans date de fermeture sont exclues des délais ; elles peuvent être en cours.",
                 "- Une anomalie statistique ne prouve ni erreur ni cause. Contrôler qualité et contexte opérationnel.",
                 "- Aucun objectif de délai/SLA n’a été fourni ; les délais sont comparés entre groupes admissibles seulement.",
                 "- Les effectifs faibles, mois incomplets et valeurs manquantes limitent les conclusions."]
report_markdown = "\n".join(report_lines)
print(report_markdown)

# Rapport Analyste BI — NYC 311

Échantillon analysé : 10,000 lignes.
Période disponible : 2026-06-04 15:32:09+00:00 à 2026-10-14 09:05:00+00:00.

## Tendances

Volume : tendance à la baisse, pente descriptive -4 783.6 demandes/mois sur 4 mois.
Médiane des délais : tendance down sur 4 mois.

## Anomalies à examiner

Détection mensuelle non exécutée : 5 mois disponibles, alors qu’il faut au moins 7 pour comparer une période à son historique. Aucune conclusion d’absence d’anomalie n’est possible.

## Délais relativement élevés

- complaint_type APPLIANCE : médiane 17.2 jours (n=52, seuil comparatif P75=6.0 jours). Ce n’est pas un constat de non-respect d’un SLA.
- complaint_type Sidewalk Condition : médiane 11.9 jours (n=80, seuil comparatif P75=6.0 jours). Ce n’est pas un constat de non-respect d’un SLA.
- complaint_type Maintenance or Facility : médiane 10.7 jours (n=84, seuil comparatif P75=6.0 jours). Ce n’est pas un constat de non-respect d’un SLA.
- complaint_type DOOR/WINDOW : méd

## 7. Tests, exports et limites

Les assertions vérifient l’ordre des périodes, les effectifs et les seuils effectivement employés, l’absence de valeurs infinies, la correspondance des fiches et l’exclusion des demandes ouvertes des délais. Un test synthétique isolé vérifie qu’un pic artificiel dans une série stable est détecté ; il n’est jamais mélangé aux données NYC 311.

Les fichiers de sortie sont créés dans `outputs/bi_analyst/`. Une table d’anomalies vide garde son schéma afin d’être consommable par les étapes suivantes. Ces contrôles vérifient la logique de référence, pas l’exhaustivité du jeu source ni une causalité métier.

In [10]:
# Test synthétique séparé : 11 mois stables et un pic artificiel au 12e mois.
# Série indépendante des données réelles, avec 100 demandes/mois puis un pic de 300.
synthetic = pd.DataFrame({"month": sum(([f"2025-{m:02d}"] * (300 if m == 12 else 100) for m in range(1,13)), [])})
synthetic_result = robust_month_anomalies(synthetic)
assert any(r["period"] == "2025-12" for r in synthetic_result), "Le test synthétique n’a pas détecté le pic artificiel."

# Vérifications sur les données réelles.
assert volume_table(df.drop(columns=["agency"], errors="ignore"), "missing_optional_column") is None
assert delay_groups(df.drop(columns=["complaint_type"], errors="ignore"), "complaint_type") is None
assert np.isfinite(trend_results[["slope_per_month", "first_value", "last_value"]].to_numpy(dtype=float)[np.isfinite(trend_results[["slope_per_month", "first_value", "last_value"]].to_numpy(dtype=float))]).all()
if monthly_volume is not None:
    assert monthly_volume["month_start"].is_monotonic_increasing
    assert int(monthly_volume["sample_count"].sum()) == int(df["month"].notna().sum())
if not anomalies.empty:
    robust_rows = anomalies[anomalies["method"] == "robust_z_mad"]
    assert (robust_rows["score"].abs() >= ROBUST_Z_THRESHOLD).all()
    assert (anomalies["n_months"] >= MIN_MONTHS_ANOMALY).all()
if "resolution_days" in df and "closed_date" in df:
    assert df.loc[~closed_mask, "resolution_days"].isna().all(), "Une demande sans fermeture a un délai utilisé."
    assert (df["resolution_days"].dropna() >= 0).all()
if len(insights_df):
    assert insights_df["insight_id"].is_unique
    assert set(insights_df["insight_type"]).issubset({"tendance", "anomalie", "délai élevé"})
assert len(synthetic_result) > 0
print("Contrôles réussis : période triée, effectifs cohérents, scores au seuil, délais limités aux demandes clôturées, insight IDs uniques, pic synthétique détecté.")

OUTPUT_DIR = PROJECT_ROOT / "outputs/bi_analyst"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
insights_path = OUTPUT_DIR / "insights.csv"
anomalies_path = OUTPUT_DIR / "anomalies.csv"
report_path = OUTPUT_DIR / "bi_analyst_report.md"
summary_path = OUTPUT_DIR / "analysis_summary.json"
insights_df.to_csv(insights_path, index=False)
anomalies.to_csv(anomalies_path, index=False)
report_path.write_text(report_markdown, encoding="utf-8")
summary = {
    "input_file": str(CLEAN_PATH), "rows": int(len(df)),
    "created_period_start": None if df["created_date"].notna().sum() == 0 else str(df["created_date"].min()),
    "created_period_end": None if df["created_date"].notna().sum() == 0 else str(df["created_date"].max()),
    "closed_with_valid_delay_n": int(valid_delay_mask.sum()),
    "monthly_volume_periods": int(volume_trend["n_periods"]),
    "anomaly_count": int(len(anomalies)), "insight_count": int(len(insights_df)),
    "anomaly_method": {"min_months": MIN_MONTHS_ANOMALY, "robust_z_threshold": ROBUST_Z_THRESHOLD,
                       "zero_mad_relative_threshold": ZERO_MAD_RELATIVE_THRESHOLD},
    "limitations": ["Échantillon uniquement", "Causes non établies", "Aucun SLA fourni", "Mois partiels possibles"]
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

# Relecture des exports : les lignes exportées doivent correspondre aux résultats calculés.
assert len(pd.read_csv(insights_path)) == len(insights_df)
assert len(pd.read_csv(anomalies_path)) == len(anomalies)
assert pd.read_csv(anomalies_path).columns.tolist() == anomaly_columns
assert json.loads(summary_path.read_text(encoding="utf-8"))["anomaly_count"] == len(anomalies)
print("Exports vérifiés :")
for path in [insights_path, anomalies_path, report_path, summary_path]:
    print(f"- {path} ({path.stat().st_size:,} octets)")

Analyse ignorée : colonne facultative 'missing_optional_column' absente.
Contrôles réussis : période triée, effectifs cohérents, scores au seuil, délais limités aux demandes clôturées, insight IDs uniques, pic synthétique détecté.


Exports vérifiés :
- D:\Analyst_Agent\.tmp_jupyter\audit_20261024\outputs\bi_analyst\insights.csv (10,311 octets)
- D:\Analyst_Agent\.tmp_jupyter\audit_20261024\outputs\bi_analyst\anomalies.csv (83 octets)
- D:\Analyst_Agent\.tmp_jupyter\audit_20261024\outputs\bi_analyst\bi_analyst_report.md (2,147 octets)
- D:\Analyst_Agent\.tmp_jupyter\audit_20261024\outputs\bi_analyst\analysis_summary.json (617 octets)


## Évaluation synthétique isolée et reproductibilité

Les scénarios artificiels sont créés en mémoire, ne touchent pas `df`, au CSV nettoyé ni aux exports métier existants. Ils évaluent hausse, baisse, absence d'anomalie, doublons contrôlés, historique court, trous/NaN, série constante, dates irrégulières, STL ignoré, reproductibilité et intégrité des sources réelles. Les seuils Z=3 et MAD=3,5 sont testés explicitement.

In [11]:
# Tests synthétiques séparés — assertions effectives, aucun fichier NYC n'est modifié.
test_results=[]
clean_hash_before_tests=hashlib.sha256(CLEAN_PATH.read_bytes()).hexdigest()
real_export_paths=[insights_path,anomalies_path,report_path,summary_path]
real_export_hashes={str(path):hashlib.sha256(path.read_bytes()).hexdigest() for path in real_export_paths if path.exists()}
def record_test(name,passed,detail):
    test_results.append({"test":name,"passed":bool(passed),"detail":str(detail)})

base=np.array([100,102,99,101,98,103,100,99,101,102,98,100],dtype=float)
z_spike=detect_zscore(base,200,period="synthetic-spike")
mad_spike=detect_mad(base,200,period="synthetic-spike")
assert z_spike["status"]=="anomaly" and z_spike["direction"]=="positive"
assert mad_spike["status"]=="anomaly" and mad_spike["direction"]=="positive"
record_test("pic_artificiel",True,{"period":z_spike["period"],"value":z_spike["observed"],"z":z_spike["score"],"robust_z":mad_spike["score"]})

z_drop=detect_zscore(base,20,period="synthetic-drop"); mad_drop=detect_mad(base,20,period="synthetic-drop")
assert z_drop["status"]=="anomaly" and z_drop["direction"]=="negative"
assert mad_drop["status"]=="anomaly" and mad_drop["direction"]=="negative"
record_test("baisse_artificielle",True,{"period":z_drop["period"],"value":z_drop["observed"],"direction":z_drop["direction"]})

normal_z=detect_zscore(base,101); normal_mad=detect_mad(base,101)
assert normal_z["status"]=="normal" and normal_mad["status"]=="normal"
record_test("serie_sans_anomalie",True,"aucune méthode n'a signalé la valeur ordinaire")

# Pic artificiel dû à copies de clés : avant déduplication l'anomalie Z existe, après déduplication elle disparaît.
synthetic_rows=[]; key_index=0
for month_number,month_volume in enumerate(base.astype(int),1):
    for _ in range(int(month_volume)):
        synthetic_rows.append({"unique_key":f"k{key_index}","month":f"2025-{month_number:02d}"}); key_index+=1
synthetic_unique=pd.DataFrame(synthetic_rows)
synthetic_duplicates=pd.concat([synthetic_unique,synthetic_unique.loc[synthetic_unique["month"]=="2025-12"].iloc[:100]],ignore_index=True)
counts_before=synthetic_duplicates.groupby("month").size()
counts_after=synthetic_duplicates.drop_duplicates("unique_key").groupby("month").size()
duplicate_peak=detect_zscore(counts_before.iloc[:-1],counts_before.iloc[-1],period="2025-12")
dedup_peak=detect_zscore(counts_after.iloc[:-1],counts_after.iloc[-1],period="2025-12")
assert duplicate_peak["status"]=="anomaly" and dedup_peak["status"]=="normal"
duplicate_test_evidence={"before_rows":int(len(synthetic_duplicates)),"after_unique_rows":int(len(synthetic_duplicates.drop_duplicates("unique_key"))),
                         "duplicate_rows":int(len(synthetic_duplicates)-len(synthetic_unique)),"z_before":duplicate_peak["score"],"z_after":dedup_peak["score"]}
record_test("pic_apparent_doublons",True,duplicate_test_evidence)
synthetic_anomaly_row=pd.DataFrame([{"status":"anomaly","direction":"positive","period":"2025-12",
    "dimension":"all","group_value":"all","observed":float(counts_before.iloc[-1]),
    "score":float(duplicate_peak["score"]),"reference_mean":float(duplicate_peak["reference_mean"]),
    "reference_n":int(duplicate_peak["reference_n"]),"reference_period":duplicate_peak["reference_period"],
    "method":"zscore","threshold":Z_THRESHOLD}])
synthetic_quality=pd.DataFrame([{"month":"2025-12","raw_duplicate_rows":duplicate_test_evidence["duplicate_rows"]}])
quality_insights,_=build_auditable_insights(synthetic_anomaly_row,pd.DataFrame(),
    {"status":"persistent","direction":"up","start_period":"2025-01","end_period":"2025-12",
     "start_value":100.0,"end_value":200.0,"change_pct":100.0,"change_abs":100.0,"required_same_direction":3,"n_periods":12},
    synthetic_quality,target_check)
assert quality_insights["data_quality_status"].str.contains("doublons_raw_période=100").any()
assert quality_insights["limitations"].str.contains("Doublons observés").any()
record_test("signal_qualite_doublons",True,"insight étiqueté comme potentiellement lié aux doublons, sans conclure à une cause réelle")

short_z=detect_zscore([1,2,3],20); short_mad=detect_mad([1,2,3],20)
constant_z=detect_zscore([5]*8,20); constant_mad=detect_mad([5]*8,20)
nan_z=detect_zscore([1,np.nan,2,3,4,5,6,7],30)
assert short_z["status"]=="insufficient_history_or_invalid_value" and short_mad["status"]=="insufficient_history_or_invalid_value"
assert constant_z["status"]=="zero_or_invalid_std" and constant_mad["status"]=="zero_or_invalid_mad"
assert nan_z["reference_n"]==7
record_test("court_nan_constant",True,{"short":short_z["status"],"constant_z":constant_z["status"],"constant_mad":constant_mad["status"],"nan_reference_n":nan_z["reference_n"]})

irregular_index=pd.period_range("2022-01",periods=25,freq="M").delete(5)
irregular=pd.Series(np.linspace(100,125,len(irregular_index)),index=irregular_index)
stl_irregular=run_stl(irregular,period=12)
stl_short=run_stl(pd.Series([1,2,3],index=pd.period_range("2024-01",periods=3,freq="M")),period=12)
assert stl_irregular["status"]=="skipped" and "irrégulier" in stl_irregular["reason"]
assert stl_short["status"]=="skipped" and stl_short["reason"] is not None
record_test("stl_historique_irrégulier_court",True,{"irregular":stl_irregular["reason"],"short":stl_short["reason"]})
regular_index=pd.period_range("2021-01",periods=36,freq="M")
regular_values=pd.Series(100+np.arange(36)*.4+8*np.sin(2*np.pi*np.arange(36)/12),index=regular_index)
regular_stl=run_stl(regular_values,period=12)
if importlib.util.find_spec("statsmodels") is not None:
    assert regular_stl["status"]=="executed" and set(["trend","seasonal","resid"]).issubset(regular_stl["decomposition"].columns)
record_test("stl_serie_reguliere",regular_stl["status"]=="executed" if importlib.util.find_spec("statsmodels") is not None else regular_stl["status"]=="skipped",regular_stl["reason"] or "décomposition synthétique produite")
residual_values=np.array([0.1,-0.1,0.05,-0.05,0.0,0.08,-0.08,0.02,-0.02,0.04,-0.04,0.0,2.0])
synthetic_decomposition=pd.DataFrame({"resid":residual_values},index=pd.period_range("2024-01",periods=len(residual_values),freq="M"))
synthetic_residual_flags=stl_residual_signals(synthetic_decomposition)
assert synthetic_residual_flags.iloc[-1]["status"]=="anomaly"
record_test("signal_résidu_STL_synthétique",True,{"period":synthetic_residual_flags.iloc[-1]["period"],"score":synthetic_residual_flags.iloc[-1]["score"]})

regular_trend=pd.DataFrame({"month":[f"2025-{m:02d}" for m in range(1,5)],
    "month_start":pd.date_range("2025-01-01",periods=4,freq="MS"),"request_count":[100,110,120,130]})
assert persistent_trend(regular_trend)["status"]=="persistent" and persistent_trend(regular_trend)["direction"]=="up"
isolated_trend=pd.DataFrame({"month":[f"2025-{m:02d}" for m in range(1,5)],
    "month_start":pd.date_range("2025-01-01",periods=4,freq="MS"),"request_count":[100,100,150,100]})
assert persistent_trend(isolated_trend)["status"]=="no_persistent_direction"
record_test("tendance_persistante_vs_isolee",True,"trois hausses successives reconnues ; variation isolée non déclarée persistante")

# Reproductibilité : mêmes valeurs, paramètres et ordre des règles produisent les mêmes résultats.
rep1=detect_mad(base,200,period="2025-12"); rep2=detect_mad(base.copy(),200,period="2025-12")
assert rep1==rep2
ins1,ev1=build_auditable_insights(advanced_anomalies,group_comparisons,persistent_volume_trend,raw_duplicate_by_month,target_check)
ins2,ev2=build_auditable_insights(advanced_anomalies,group_comparisons,persistent_volume_trend,raw_duplicate_by_month,target_check)
pd.testing.assert_frame_equal(ins1,ins2); pd.testing.assert_frame_equal(ev1,ev2)
record_test("reproductibilite",True,"scores, règles, preuves et observations identiques")

# Aucun test synthétique ne modifie les objets métier ni le fichier nettoyé.
assert len(df)==len(df_source) and df_source.equals(pd.read_csv(CLEAN_PATH,low_memory=False))
assert hashlib.sha256(CLEAN_PATH.read_bytes()).hexdigest()==clean_hash_before_tests
assert all(hashlib.sha256(Path(path).read_bytes()).hexdigest()==digest for path,digest in real_export_hashes.items())
record_test("pas_de_fuite_vers_reel",True,"DataFrame réel et CSV source inchangés")

# Vérifier périodes triées, mesures finies, effectifs et cibles métier non configurées.
if len(trend_analysis): assert trend_analysis["month_start"].is_monotonic_increasing
if len(group_comparisons): assert (group_comparisons["sample_size"]>0).all()
assert target_check["status"]=="cible métier non configurée"
record_test("contrôles_sorties_réelles",True,{"months":len(trend_analysis),"groups":len(group_comparisons),"target":target_check["status"]})

test_results_df=pd.DataFrame(test_results)
assert test_results_df["passed"].all()
display(test_results_df)

# Visualisations analytiques avec réel et synthétique explicitement séparés.
if len(anomaly_comparison):
    fig,ax=plt.subplots(figsize=(9,4)); ax.plot(anomaly_comparison["period"],anomaly_comparison["score"],marker="o",label="Scores calculables")
    ax.axhline(Z_THRESHOLD,color="red",linestyle="--",label=f"Z +{Z_THRESHOLD}"); ax.axhline(-Z_THRESHOLD,color="red",linestyle="--",label=f"Z -{Z_THRESHOLD}")
    ax.set(title="NYC 311 — scores Z des mois évaluables",xlabel="Mois analysé",ylabel="Z-score"); ax.legend(); plt.xticks(rotation=45); plt.tight_layout(); plt.show()
if stl_result["status"]=="executed":
    stl_decomposition.plot(subplots=True,figsize=(10,8),title="NYC 311 — décomposition STL"); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(figsize=(9,4)); ax.plot(range(len(base)),base,label="Historique synthétique"); ax.scatter([len(base)],[200],color="red",label="Pic injecté")
ax.set(title="TEST SYNTHÉTIQUE — pic artificiel (pas de données NYC 311)",xlabel="Période synthétique",ylabel="Demandes simulées"); ax.legend(); plt.tight_layout(); plt.show()

# Exports enrichis, tout en conservant les quatre chemins historiques.
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
group_path=OUTPUT_DIR/"group_comparisons.csv"; trend_path=OUTPUT_DIR/"trend_analysis.csv"
stl_path=OUTPUT_DIR/"stl_decomposition.csv"; tests_path=OUTPUT_DIR/"test_results.json"
evidence_path=OUTPUT_DIR/"insight_evidence.csv"; comparison_path=OUTPUT_DIR/"anomaly_method_comparison.csv"
group_comparisons.to_csv(group_path,index=False); trend_analysis.to_csv(trend_path,index=False)
if stl_result["status"]=="executed": stl_decomposition.to_csv(stl_path,index=True)
test_results_path_json={"all_passed":bool(test_results_df["passed"].all()),"tests":test_results,
                        "note":"Synthetic tests are in-memory and separate from NYC 311."}
tests_path.write_text(json.dumps(test_results_path_json,ensure_ascii=False,indent=2,default=str),encoding="utf-8")
evidence_registry.to_csv(evidence_path,index=False); anomaly_comparison.to_csv(comparison_path,index=False)
if len(stl_residual_comparison): stl_residual_comparison.to_csv(OUTPUT_DIR/"stl_residual_signals.csv",index=False)
rule_insights.to_csv(insights_path,index=False)
advanced_anomalies.to_csv(anomalies_path,index=False)

enhanced_summary={**summary,"methods_executed":{"zscore":("evaluated" if len(anomaly_comparison) else f"skipped: {len(trend_monthly_volume) if trend_monthly_volume is not None else 0} month(s); need 7"),
    "mad":("evaluated" if len(anomaly_comparison) else f"skipped: {len(trend_monthly_volume) if trend_monthly_volume is not None else 0} month(s); need 7"),"stl":stl_result["status"],
    "stl_reason":stl_result["reason"],"group_comparisons":int(len(group_comparisons)),
    "trend_periods":int(len(trend_analysis)),"targets":target_check["status"]},
    "anomaly_method":{"zscore_threshold":Z_THRESHOLD,"mad_threshold":MAD_THRESHOLD,"min_reference_periods":MIN_REFERENCE_N,
                       "zero_dispersion":"no score and no alert"},
    "anomaly_count":int(len(advanced_anomalies)),"insight_count":int(len(rule_insights)),
    "anomaly_config":{"z_threshold":Z_THRESHOLD,"mad_threshold":MAD_THRESHOLD,"min_reference_n":MIN_REFERENCE_N},
    "rule_version":RULE_VERSION,"quality_report_status":quality_report_status,"raw_quality_status":raw_quality_status,
    "synthetic_tests_all_passed":bool(test_results_df["passed"].all()),"synthetic_test_count":len(test_results),
    "sample_design":None if sample_metadata is None else sample_metadata.get("sample_method"),
    "sample_monthly_counts":None if sample_metadata is None else sample_metadata.get("sample_monthly_counts"),
    "population_monthly_counts":None if sample_metadata is None else sample_metadata.get("population_monthly_counts"),
    "partial_months":sorted(partial_months)}
summary_path.write_text(json.dumps(enhanced_summary,ensure_ascii=False,indent=2,default=str),encoding="utf-8")

report_lines += ["", "## Méthodes statistiques et évaluation", "",
                 f"- Z-score classique : seuil ±{Z_THRESHOLD}; MAD robuste : seuil ±{MAD_THRESHOLD}; référence minimale {MIN_REFERENCE_N} périodes antérieures.",
                 f"- Anomalies réelles : {len(anomaly_comparison)} comparaisons exécutées; {len(advanced_anomalies)} signal(s). {'Historique trop court : détection non exécutée.' if not len(anomaly_comparison) else 'Comparaisons calculées.'}",
                 f"- Tendance persistante : {persistent_volume_trend['status']} ({persistent_volume_trend.get('n_periods',0)} périodes).",
                 f"- STL : {stl_result['status']} — {stl_result['reason'] or 'décomposition calculée'}.",
                 f"- Cible métier : {target_check['status']}. Aucun SLA n’a été inventé.",
                 f"- Tests synthétiques : {len(test_results)} exécutés, tous réussis : {bool(test_results_df['passed'].all())}. Ces résultats ne sont pas des résultats NYC 311.",
                 f"- Rapport qualité externe : {quality_report_status}; source brute : {raw_quality_status}.",
                 f"- Base des volumes mensuels : {monthly_volume['count_basis'].iloc[0] if monthly_volume is not None else 'indisponible'}; mois partiels exclus des tendances : {sorted(partial_months)}. Les comparaisons de catégories/délais utilisent des blocs temporels répartis, non un tirage aléatoire, donc un biais de sélection reste possible.",
                 f"- Insights auditables produits : {len(rule_insights)}; preuves : {len(evidence_registry)}; règle {RULE_VERSION}."]
report_markdown="\n".join(report_lines)
report_path.write_text(report_markdown,encoding="utf-8")

# Vérification des exports réels et du schéma des anomalies, y compris dataframe vide.
assert len(pd.read_csv(insights_path))==len(rule_insights)
assert len(pd.read_csv(anomalies_path))==len(advanced_anomalies)
assert len(pd.read_csv(group_path))==len(group_comparisons)
assert len(pd.read_csv(trend_path))==len(trend_analysis)
assert json.loads(tests_path.read_text(encoding="utf-8"))["all_passed"]
print("Exports renforcés vérifiés :",[str(x) for x in [insights_path,anomalies_path,report_path,summary_path,group_path,trend_path,tests_path,evidence_path,comparison_path]])

,test,passed,detail
0,pic_artificiel,True,"{'period': 'synthetic-spike', 'value': 200.0, 'z': 62.24436215006967, 'robust_z': 67.45}"
1,baisse_artificielle,True,"{'period': 'synthetic-drop', 'value': 20.0, 'direction': 'negative'}"
2,serie_sans_anomalie,True,aucune méthode n'a signalé la valeur ordinaire
3,pic_apparent_doublons,True,"{'before_rows': 1303, 'after_unique_rows': 1203, 'duplicate_rows': 100, 'z_before': 59.405880609804896, 'z_after': -..."
4,signal_qualite_doublons,True,"insight étiqueté comme potentiellement lié aux doublons, sans conclure à une cause réelle"
5,court_nan_constant,True,"{'short': 'insufficient_history_or_invalid_value', 'constant_z': 'zero_or_invalid_std', 'constant_mad': 'zero_or_inv..."
6,stl_historique_irrégulier_court,True,"{'irregular': 'index mensuel irrégulier ou périodes manquantes ; aucune interpolation appliquée', 'short': 'historiq..."
7,stl_serie_reguliere,True,décomposition synthétique produite
8,signal_résidu_STL_synthétique,True,"{'period': '2025-01', 'score': np.float64(29.977777777777778)}"
9,tendance_persistante_vs_isolee,True,trois hausses successives reconnues ; variation isolée non déclarée persistante


Exports renforcés vérifiés : ['D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\insights.csv', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\anomalies.csv', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\bi_analyst_report.md', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\analysis_summary.json', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\group_comparisons.csv', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\trend_analysis.csv', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\test_results.json', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\insight_evidence.csv', 'D:\\Analyst_Agent\\.tmp_jupyter\\audit_20261024\\outputs\\bi_analyst\\anomaly_method_comparison.csv']


C:\Users\HP\AppData\Local\Temp\ipykernel_8264\1738156756.py:121: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(title="TEST SYNTHÉTIQUE — pic artificiel (pas de données NYC 311)",xlabel="Période synthétique",ylabel="Demandes simulées"); ax.legend(); plt.tight_layout(); plt.show()


## 8. Synthèse locale avec Llama

Les KPI, seuils, anomalies et recommandations opérationnelles sont calculés et exportés par le code déterministe. Llama 3.2 3B ne reçoit que les insights agrégés pour rédiger un court résumé qualitatif, sans chiffres. Si le modèle n’est pas joignable ou produit une sortie douteuse, cette synthèse est omise et le rapport BI déterministe reste disponible.

In [12]:
print('LLM narrative skipped during deterministic audit')

LLM narrative skipped during deterministic audit


## Comment exécuter

1. Depuis la racine du projet, exécuter `notebooks/01_etl_kpi.ipynb` afin de produire les données nettoyées.
2. Vérifier qu’Ollama est lancé et que `llama3.2:3b` est installé (`ollama list`).
3. Ouvrir `notebooks/02_bi_analyst.ipynb` dans Jupyter ou VS Code et exécuter toutes les cellules dans l’ordre.
4. Vérifier la création des fichiers dans `outputs/bi_analyst/` et consulter `bi_analyst_report.md`.

Le Notebook 1 n’a pas été modifié. Si Ollama est indisponible, les exports déterministes restent produits et la synthèse Llama est signalée comme indisponible. Cette version Analyste BI reste descriptive : une période courte peut ne permettre ni tendance ni détection statistique, et toute explication causale nécessite des données/contextes complémentaires.